## **Imports**

In [1]:
%load_ext autoreload
%autoreload 2
import torch
import dill
import hydra
from tqdm.auto import tqdm
import numpy as np
import collections
from skvideo.io import vwrite
from omegaconf import OmegaConf

from diffusion_policy.workspace.base_workspace import BaseWorkspace
from diffusion_policy.policy.base_lowdim_prob_policy import BaseLowdimProbPolicy
from diffusion_policy.env.pusht.pusht_keypoints_env import PushTKeypointsEnv 
from diffusion_policy.common.pytorch_util import dict_apply

pygame 2.1.2 (SDL 2.0.16, Python 3.9.18)
Hello from the pygame community. https://www.pygame.org/contribute.html


## **Configuration**

In [16]:
device="cuda:0"
n_policy_regeneration=20
stochastic = False
seed = 100055
override = ["policy.eta=1.0"]

## **Load checkpoint**

In [ ]:
#load Standard Diffusion Policy checkpoint
checkpoint_DP ="data/outputs/2026.02.19/13.50.47_train_diffusion_unet_lowdim_pusht_lowdim/checkpoints/epoch=4350-test_mean_score=0.900.ckpt"
payload = torch.load(open(checkpoint_DP, 'rb'), pickle_module=dill)
print ("payload keys:", payload.keys())
cfg = payload['cfg']

# apply overrides (if any)
if override:
    override_cfg = OmegaConf.from_dotlist(override)
    cfg = OmegaConf.merge(cfg, override_cfg)
    
cls = hydra.utils.get_class(cfg._target_)
workspace = cls(cfg)
workspace: BaseWorkspace
workspace.load_payload(payload, exclude_keys=None, include_keys=None)

# get policy from workspace
policy_DP = workspace.model
if cfg.training.use_ema:
    policy_DP = workspace.ema_model

device = torch.device(device)
policy_DP.to(device)
policy_DP.eval()

#load Bayesian Diffusion Policy checkpoint
checkpoint_BDP ="data/outputs/2026.02.19/10.22.26_train_prob_diffusion_unet_lowdim_pusht_lowdim/checkpoints/epoch=4450-test_mean_score=0.922.ckpt"
payload = torch.load(open(checkpoint_BDP, 'rb'), pickle_module=dill)
print ("payload keys:", payload.keys())
cfg = payload['cfg']

# apply overrides (if any)
if override:
    override_cfg = OmegaConf.from_dotlist(override)
    cfg = OmegaConf.merge(cfg, override_cfg)
    
cls = hydra.utils.get_class(cfg._target_)
workspace = cls(cfg)
workspace: BaseWorkspace
workspace.load_payload(payload, exclude_keys=None, include_keys=None)

# get policy from workspace
policy_BDP = workspace.model
if cfg.training.use_ema:
    policy_BDP = workspace.ema_model

device = torch.device(device)
policy_BDP.to(device)
policy_BDP.eval()




payload keys: dict_keys(['cfg', 'state_dicts', 'pickles'])
payload keys: dict_keys(['cfg', 'state_dicts', 'pickles'])


DiffusionUnetLowdimProbPolicy(
  (model): BayesianConditionalUnet1D(
    (mid_modules): ModuleList(
      (0-1): 2 x ProbConditionalResidualBlock1D(
        (blocks): ModuleList(
          (0-1): 2 x ProbConv1dBlock(
            (block): Sequential(
              (0): ProbConv1d(
                (weight): Gaussian()
                (bias): Gaussian()
                (weight_prior): Gaussian()
                (bias_prior): Gaussian()
              )
              (1): GroupNorm(8, 1024, eps=1e-05, affine=True)
              (2): Mish()
            )
          )
        )
        (cond_encoder): Sequential(
          (0): Mish()
          (1): Linear(in_features=296, out_features=2048, bias=True)
          (2): Rearrange('batch t -> batch t 1')
        )
        (residual_conv): Identity()
      )
    )
    (dropout): Identity()
    (diffusion_step_encoder): Sequential(
      (0): SinusoidalPosEmb()
      (1): Linear(in_features=256, out_features=1024, bias=True)
      (2): Mish()
      

## **Inference Standard Diffusion Policy**

In [19]:
# configure pusht environment
env = PushTKeypointsEnv(render_size=512, reset_to_state=[300,200,200,400,0*-np.pi/2])
env.seed(seed)

# get first observation
obs = env.reset()

# keep a queue of last 2 steps of observations
obs_deque = collections.deque([obs] * cfg.n_obs_steps, maxlen=cfg.n_obs_steps)

# save visualization and rewards
imgs = [env.render(mode="rgb_array")]
max_steps=300
rewards = list()
done = False
step_idx = 0

# Same noise sequance during inference 
torch.manual_seed(42)
np.random.seed(42)

with tqdm(total=max_steps, desc="Eval PushTStateEnv") as pbar:
    while not done:
        pred_actions = []
        Do = obs.shape[-1] // 2

        obs_seq = np.stack(obs_deque)
        obs_seq = np.expand_dims(obs_seq,axis=0)

        # create obs dict
        np_obs_dict = {
            # handle n_latency_steps by discarding the last n_latency_steps
            'obs': obs_seq[...,:cfg.n_obs_steps,:Do].astype(np.float32),
            'obs_mask': obs_seq[...,:cfg.n_obs_steps,Do:] > 0.5
        }

        # device transfer
        obs_dict = dict_apply(np_obs_dict, 
            lambda x: torch.from_numpy(x).to(
                device=device))

        # run policy: it is either standard DP or Bayesian DP
        with torch.no_grad():
            for i in range(n_policy_regeneration):
                action_dict = policy_DP.predict_action(obs_dict)
            
                # device_transfer
                np_action_dict = dict_apply(action_dict,
                    lambda x: x.detach().to('cpu').numpy())
                
                # handle latency_steps, we discard the first n_latency_steps actions
                # to simulate latency
                action = np_action_dict['action'][:,cfg.n_latency_steps:]
                pred_actions.append(action[0])
            
        # random selection of the predicted actions
        idx = np.random.randint(0, n_policy_regeneration)
        pred_actions = np.stack(pred_actions, axis=0)
        pred_action = pred_actions[idx]

        # execute action_horizon number of steps
        # without replanning
        for i in range(len(pred_action)):
            # step env
            obs, reward, done, info = env.step(pred_action[i])
            # save observations
            obs_deque.append(obs)
            # and reward/vis
            rewards.append(reward)
            if i==0:
                for _ in range (n_policy_regeneration):
                    imgs.append(env.render(mode="rgb_array", pred_action= pred_action, pred_actions=pred_actions ))
            else:
                imgs.append(env.render(mode="rgb_array", pred_action=pred_action, pred_actions=None ))
            # update progress bar
            step_idx += 1
            pbar.update(1)
            pbar.set_postfix(reward=reward)
            if step_idx > max_steps:
                done = True
            if done:
                break

            past_action = action
            pbar.update(1)
            pbar.set_postfix(reward=reward)

average_max_reward = max(rewards)

# print out the maximum target coverage
print("test_mean_score: ", average_max_reward)
vwrite("pusht_DP-eta="+str(cfg.policy.eta)+".mp4", imgs)



Eval PushTStateEnv:   0%|          | 0/300 [00:00<?, ?it/s]

test_mean_score:  1.0


## **Inference Bayesian Diffusion Policy**

In [23]:
# configure pusht environment
env = PushTKeypointsEnv(render_size=512, reset_to_state=[300,200,200,280,0.5*np.pi/2])
env.seed(seed)

# get first observation
obs = env.reset()

# keep a queue of last 2 steps of observations
obs_deque = collections.deque([obs] * cfg.n_obs_steps, maxlen=cfg.n_obs_steps)

# save visualization and rewards
imgs = [env.render(mode="rgb_array")]
max_steps=300
rewards = list()
done = False
step_idx = 0

# Same noise sequance during inference 
torch.manual_seed(42)
np.random.seed(42)

with tqdm(total=max_steps, desc="Eval PushTStateEnv") as pbar:
    while not done:
        pred_actions = []
        Do = obs.shape[-1] // 2

        obs_seq = np.stack(obs_deque)
        obs_seq = np.expand_dims(obs_seq,axis=0)

        # create obs dict
        np_obs_dict = {
            # handle n_latency_steps by discarding the last n_latency_steps
            'obs': obs_seq[...,:cfg.n_obs_steps,:Do].astype(np.float32),
            'obs_mask': obs_seq[...,:cfg.n_obs_steps,Do:] > 0.5
        }

        # device transfer
        obs_dict = dict_apply(np_obs_dict, 
            lambda x: torch.from_numpy(x).to(
                device=device))

        # run policy: it is either standard DP or Bayesian DP
        with torch.no_grad():
            for i in range(n_policy_regeneration):
                action_dict = policy_BDP.predict_action(obs_dict, stochastic=stochastic)

                # device_transfer
                np_action_dict = dict_apply(action_dict,
                    lambda x: x.detach().to('cpu').numpy())
                
                # handle latency_steps, we discard the first n_latency_steps actions
                # to simulate latency
                action = np_action_dict['action'][:,cfg.n_latency_steps:]
                pred_actions.append(action[0])
            
        # random selection of the predicted actions
        idx = np.random.randint(0, n_policy_regeneration)
        pred_actions = np.stack(pred_actions, axis=0)
        pred_action = pred_actions[idx]

        # execute action_horizon number of steps
        # without replanning
        for i in range(len(pred_action)):
            # step env
            obs, reward, done, info = env.step(pred_action[i])
            # save observations
            obs_deque.append(obs)
            # and reward/vis
            rewards.append(reward)
            if i==0:
                for _ in range (n_policy_regeneration):
                    imgs.append(env.render(mode="rgb_array", pred_action= pred_action, pred_actions=pred_actions ))
            else:
                imgs.append(env.render(mode="rgb_array", pred_action=pred_action, pred_actions=None ))
            # update progress bar
            step_idx += 1
            pbar.update(1)
            pbar.set_postfix(reward=reward)
            if step_idx > max_steps:
                done = True
            if done:
                break

            past_action = action
            pbar.update(1)
            pbar.set_postfix(reward=reward)

average_max_reward = max(rewards)

# print out the maximum target coverage
print("test_mean_score: ", average_max_reward)
vwrite("pusht_BDP-stochastic="+str(stochastic)+"-eta="+str(cfg.policy.eta)+".mp4", imgs)

Eval PushTStateEnv:   0%|          | 0/300 [00:00<?, ?it/s]

test_mean_score:  1.0
